In [1]:
import re, html, pandas as pd, spacy

df = pd.read_csv("../data/processed/eda_clean.csv")
print(df.shape)

nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])
KEEP = {"not", "no", "never", "nor", "n't", "cannot"}   # negations matter!

def basic_clean(t):
    t = html.unescape(t)                  # &quot; -> "  and  &amp; -> &
    t = t.lower()
    t = re.sub(r"<.*?>", " ", t)          # html tags
    t = re.sub(r"http\S+", " ", t)        # urls
    t = re.sub(r"[^a-z\s']", " ", t)      # keep letters and apostrophes
    return re.sub(r"\s+", " ", t).strip()

def lemmatize(texts):
    out = []
    for doc in nlp.pipe(texts, batch_size=500):
        toks = [tok.lemma_ for tok in doc
                if tok.lower_ in KEEP or (not tok.is_stop and len(tok) > 2)]
        out.append(" ".join(toks))
    return out

(79809, 6)


In [2]:
sample = df["text"].head(5).tolist()
for raw, clean in zip(sample, lemmatize([basic_clean(t) for t in sample])):
    print("RAW:  ", raw[:200])
    print("CLEAN:", clean[:200], "\n")

RAW:   Basically, this book shows that most of America's great statesmen from the Revolutionary war to the Kennedy years were not overly fond of Blacks and did not beleive in the intelligence fo Black people
CLEAN: basically book show america great statesman revolutionary war kennedy year not overly fond black not beleive intelligence black people ability assimilate american society good provide interesting stat 

RAW:   The Two Towers includes book III and IV in the Lord of the Rings cycle. There is a ring that makes the wearer invisible, but it also corrupts that person. The ring also gives the bearer unimaginable p
CLEAN: tower include book iii lord ring cycle ring make wearer invisible corrupt person ring give bearer unimaginable power fantasy world tower take place evil ruler sauron prevent ring fall hand sauron dest 

RAW:   Having never read The Count of Monte Cristo and only faintly recalling a movie of some years ago, I was prepared for a swashbuckling epic of swordplay and de

In [3]:
from nltk.stem import PorterStemmer
ps = PorterStemmer()
for w in ["loving", "loved", "beautiful", "endings", "better"]:
    print(w, "-> stem:", ps.stem(w))
print([t.lemma_ for t in nlp("loving loved beautiful endings better")])

loving -> stem: love
loved -> stem: love
beautiful -> stem: beauti
endings -> stem: end
better -> stem: better
['love', 'love', 'beautiful', 'ending', 'well']


In [4]:
df["clean"] = lemmatize(df["text"].map(basic_clean))
df = df[df["clean"].str.split().str.len() >= 3]      # drop reviews that became empty
df.to_csv("../data/processed/clean.csv", index=False)
print(df.shape)
df[["text", "clean"]].head()

(79788, 7)


,text,clean
0,"Basically, this book shows that most of Americ...",basically book show america great statesman re...
1,The Two Towers includes book III and IV in the...,tower include book iii lord ring cycle ring ma...
2,Having never read The Count of Monte Cristo an...,have never read count monte cristo faintly rec...
3,What a fun book! I had a great time reading th...,fun book great time read chuckle way part book...
4,Many times one reads a book for an intricate p...,time read book intricate plot brilliant charac...
